# Setup

### Instalation (Kafka)
Reference video: [Guides](https://youtu.be/LX5LKBYHmyU?si=hJaSakdetjpwTCCC)

Step 1: Install Java

Kafka requires Java 17 or higher. Check if Java is installed by running java -version. If it is not, install it using your system's package manager (e.g., sudo apt install openjdk-17-jdk on Ubuntu).

Step 2: Download and Extract Kafka

Go to the Apache Kafka Downloads Page and download the latest binary release (e.g., Scala 2.13).Extract the archive and navigate to the directory:
```bash
tar -xzf kafka_2.13-4.2.0.tgz
cd kafka_2.13-4.2.0
```
(Note: Replace 4.2.0 with the exact downloaded version).

### Instalation (Mongodb)
Guidance video: [Guides](https://youtu.be/HT_TUkqK9Wg?si=XwWrt3Brl3fX0a9S)

Step 1: First we need to install an extension in VS code.
"MongoDB for VS code"

Step 2: Connect to MongoDB.
default is a local MongoDB server at mongodb://127.0.0.1:27017
Note: Make sure your MongoDB server (mongod.exe) is running if you are connecting to a local MongoDB server.

Step 3: Let's attach a MongoDB shell to the active connection.
Note: Make sure the MongoDB shell (mongo or mongosh) is installed and is on your path.

Step 4: MongoDB commands
There are MongoDB specific commands available in the VS Code Command Palette (Ctrl+Shift+P) as well as
through Explorer context menus.

Step 5: Using Playground inside visual studio code. (quick and easy way to query MongoDB).
Playgrounds let you create, run, and save MongoDB commands from a VS Code editor.

### Create Virtual environment
create venv in project folder
```bash
# Go to your project folder
cd "path/to/your/streaming/project"

# Create a virtual environment using 3.12
py -3.12 -m venv venv

# Activate it
venv\Scripts\activate

# install relevant modules (important ! )
```


# Prerequisite (Before impl)

## mongodb

Since the server is actively running in this command prompt window, it is "occupying" this terminal. To run commands to deep clean (drop) your databases, you have two options. You can either **leave this window open** and open a **new** Command Prompt window, or use a graphical tool like MongoDB Compass.

Here is the cleanest way to wipe the data using the command line:

### Step 1: Open a New Command Prompt

Leave your current terminal running (if you close it, MongoDB will shut down). Open a **second** Command Prompt window.

### Step 2: Connect via mongosh

In the new window, type the following command to open the MongoDB Shell:

```cmd
mongosh

```

*(Note: If you are using an older installation, the command might be mongo, but modern versions use mongosh)*.

### Step 3: Deep Clean Your Data

Once you are connected to the shell (you'll see a prompt changing to something like test>), you can clean your data using these commands:

* **To wipe out a specific database completely:**
```javascript
use your_database_name
db.dropDatabase()


```



```
*   **To clean just a single collection (table) inside a database without losing the database itself:**
    ```javascript
    use your_database_name
    db.your_collection_name.drop()
    

```

---

### Alternative: The "Nuclear" Option (Fresh Start)

If you are doing local development and want to delete **absolutely everything** to start completely fresh from scratch:

1. Go back to your original Command Prompt window and press **Ctrl + C** to safely shut down the MongoDB server.
2. Open your Windows File Explorer and navigate to **C:\data\db**.
3. Select everything inside that folder and delete it.
4. Run your original startup command again:
```cmd
mongod --dbpath "C:\data\db"


```
---
## kafka


### What is the kraft-combined-logs folder?

In KRaft mode, Kafka merges its two main responsibilities into a single node setup: acting as a **Broker** (handling your message data) and a **Controller** (managing cluster metadata and coordination). Because it acts as both, it creates a **"combined"** storage directory.

Inside kraft-combined-logs, Kafka stores:

* **Your Data Topics:** Every message, event, partition, and offset you produce to Kafka.
* **Cluster Metadata (__cluster_metadata):** The internal ledger that tracks which brokers exist, what topics exist, and who is in charge.
* **The Cluster ID:** A hidden unique tracking file (meta.properties) that binds your data folder to a specific cluster generation token.

---

### Why do you need to remove it before starting Kafka again?

If you are developing locally and trying to reset Kafka, simply restarting the command script often fails with an error. This happens for two main reasons:

1. **The Cluster ID Mismatch (Most Common Error):**
Every time you run the Kafka setup command to format your storage directory, it generates a brand new **Cluster ID**. If you try to boot Kafka using an *old* kraft-combined-logs folder with a *new* Cluster ID, Kafka will panic and crash because the IDs do not match.
2. **Corrupted Metadata Logs:**
If your previous Kafka instance crashed, was forcefully shut down (Ctrl + C or terminal closure), or has lingering corrupted state records, it will prevent a clean reboot.

> To Kafka, an old data directory looks like a cluster that belongs to someone else. Clearing it allows Kafka to initialize a brand-new, completely clean environment.

---

### How to safely remove and re-apply Kafka

Follow these steps to clean out the logs and start Kafka back up with a clean slate.

1. **Stop Kafka completely:** Required.
Close any active command prompts running Kafka, or press **Ctrl + C** in those windows and wait for the processes to fully shut down.


2. **Delete the log directory:** Deep Clean.
Navigate to your configured Kafka data directory (often found in C:\tmp\kraft-combined-logs or inside your extracted Kafka folder under /data/). Delete the entire **kraft-combined-logs** folder.


3. **Generate a fresh Cluster ID:** Configuration.
Open a command prompt inside your Kafka directory and generate a brand-new cluster UUID string by running:

```cmd
.\bin\windows\kafka-storage.bat random-uuid
.\bin\windows\kafka-storage.bat random-uuid
```

```cmd
    .\bin\windows\kafka-storage.bat format -t YOUR_UUID_HERE -c .\config\kraft\server.properties
    .\bin\windows\kafka-storage.bat format -t YOUR_UUID_HERE -c .\config\kraft\server.properties

```

This recreates a completely fresh, uncorrupted version of the kraft-combined-logs directory.


4. **Launch Kafka:** Ready.
Now you can safely boot Kafka back up with a clean slate:

```cmd
.\bin\windows\kafka-server-start.bat .\config\kraft\server.properties
.\bin\windows\kafka-server-start.bat .\config\kraft\server.properties

```




# Implementation
### Start Kafka & MongoDB (Before Running)
Terminal 1 - Start Kafka (using KRaft):

```bash
# Navigate to your Kafka installation 
cd C:\kafka

set KAFKA_HEAP_OPTS=-Xmx1G -Xms1G

# Generate cluster ID (only do this once) 
.\bin\windows\kafka-storage.bat random-uuid 
# This outputs something like: MkVmODQ5NTcwMjJkNDMyYzA

# Format storage with KRaft (only do this once, saves the cluster ID) 
.\bin\windows\kafka-storage.bat format -t MkVmODQ5NTcwMjJkNDMyYzA -c .\config\server.properties

# Start Kafka broker
.\bin\windows\kafka-server-start.bat .\config\server.properties
```

Terminal 2 - Start MongoDB:
```bash
mongod --dbpath "C:\data\db" 
# or if installed as a service 
net start MongoDB
```

### Create Kafka Topics (Terminal 3)
```bash
cd C:\kafka 
 
# Create topics for each camera 
.\bin\windows\kafka-topics.bat --create --bootstrap-server localhost:9092 --topic camera-events-camera-1 --partitions 1 --replication-factor 1 
 
.\bin\windows\kafka-topics.bat --create --bootstrap-server localhost:9092 --topic camera-events-camera-2 --partitions 1 --replication-factor 1 
 
.\bin\windows\kafka-topics.bat --create --bootstrap-server localhost:9092 --topic camera-events-camera-3 --partitions 1 --replication-factor 1 
 
.\bin\windows\kafka-topics.bat --create --bootstrap-server localhost:9092 --topic speed-violations --partitions 1 --replication-factor 1
```

### Run Camera Producers (Terminal 4, 5 and 6)
Note: navigate to src folder.
```bash
# For camera A
python producer_utils.py camera_a
# For camera B
python producer_utils.py camera_b
# For camera C
python producer_utils.py camera_c
```
